# Competencia: ¿qué árbol cubre el terreno?

**Sesión 2 · IA Intermedio · EstudIA 2026**

Cada fila describe una parcela de 30 × 30 metros del bosque nacional Roosevelt, en Colorado (Estados
Unidos): su altura, su pendiente, qué tan lejos está del agua o de una vía, cuánta sombra recibe, el
área protegida y el tipo de suelo. Tu red tiene que predecir **qué árbol cubre la parcela**.

| Etiqueta | Cobertura |
|---|---|
| `picea_abeto` | Pícea y abeto (*spruce/fir*) |
| `pino_contorta` | Pino contorta (*lodgepole pine*) |
| `pino_ponderosa` | Pino ponderosa |
| `alamo_sauce` | Álamo y sauce (*cottonwood/willow*) |
| `alamo_temblon` | Álamo temblón (*aspen*) |
| `abeto_douglas` | Abeto de Douglas |
| `krummholz` | Krummholz: árboles enanos y retorcidos del límite del bosque |

**Reglas**

1. Usa una **red neuronal en Keras**. Todo lo demás lo puedes cambiar.
2. Tienes 14.000 parcelas con etiqueta (`entrenamiento`) y 4.900 sin etiqueta (`prueba`). Separa tu
   propia validación: no tienes las etiquetas de prueba.
3. Envía tus predicciones todas las veces que quieras (una vez cada 10 segundos). Gana la mayor
   **exactitud**. Las siete clases tienen el mismo número de parcelas.
4. La tabla en vivo mide tu exactitud con **la mitad** de las filas de prueba. Al final, el profesor
   revela el resultado con **la otra mitad**: ese decide la competencia.

**Cómo empezar.** *Archivo › Guardar una copia en Drive*. Ejecuta todo el notebook tal como está
(*Entorno de ejecución › Ejecutar todas*). Cuando te pida la clave, cópiala de la página de la
competencia en la plataforma y pégala. Así haces tu primer envío. Después, mejora tu modelo.

In [ ]:
# Preparación: librerías y datos. Ejecútala una vez.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s02")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn.model_selection import train_test_split

entrenamiento = pd.read_csv(f"{ARCHIVOS}/cobertura_entrenamiento.csv")
prueba = pd.read_csv(f"{ARCHIVOS}/cobertura_prueba.csv")
CLASES = sorted(entrenamiento["cobertura"].unique())
print(f"Listo: {len(entrenamiento):,} parcelas de entrenamiento y {len(prueba):,} de prueba.")

In [ ]:
# @title Función para enviar tus predicciones (no hace falta cambiarla)
import getpass, json, urllib.error, urllib.request

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")

def enviar(ids, predicciones, nota=""):
    """Envía tus predicciones a la plataforma del curso y muestra tu exactitud en la tabla.

    ids: la columna "id" del archivo de prueba. predicciones: una etiqueta por fila, como
    "pino_contorta". nota: un texto corto para recordar qué probaste (opcional)."""
    global _CLAVE
    if not _CLAVE:
        _CLAVE = getpass.getpass("Pega tu clave (plataforma › Competencia › Copiar) y oprime Enter: ").strip()
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "predicciones": [str(p) for p in predicciones],
                         "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s02/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""  # la próxima vez la pide de nuevo
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    print(f"Envío {res['envio']} recibido. Exactitud en la tabla: {res['exactitud']:.1%}.")
    print(f"Tu mejor resultado: {res['mejor']:.1%} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}).")

In [ ]:
# @title Función para dibujar las curvas de entrenamiento
def curvas(historia, titulo):
    """Pérdida y exactitud por época, en entrenamiento y en validación."""
    h = historia.history
    epocas = range(1, len(h["loss"]) + 1)
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
    for eje, metrica, nombre in ((ejes[0], "loss", "Pérdida"), (ejes[1], "accuracy", "Exactitud")):
        eje.plot(epocas, h[metrica], label="Entrenamiento", color="#000000", lw=2)
        eje.plot(epocas, h["val_" + metrica], label="Validación", color="#CB318E", lw=2)
        eje.set_xlabel("Época"); eje.set_title(nombre); eje.grid(alpha=0.3)
        eje.spines[["top", "right"]].set_visible(False)
    mejor = int(np.argmin(h["val_loss"]))
    ejes[0].axvline(mejor + 1, color="#7FC200", ls="--", lw=1.5, label=f"Menor pérdida de validación (época {mejor + 1})")
    ejes[0].legend(frameon=False, fontsize=9)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); plt.show()
    return mejor + 1

---
## 1. Los datos

| Columna | Qué mide |
|---|---|
| `elevacion` | Altura sobre el nivel del mar, en metros |
| `orientacion` | Hacia dónde mira la ladera, en grados (0 a 360, 0 = norte) |
| `pendiente` | Inclinación del terreno, en grados |
| `dist_horiz_agua`, `dist_vert_agua` | Distancia horizontal y vertical al agua más cercana, en metros |
| `dist_horiz_via` | Distancia horizontal a la vía más cercana, en metros |
| `sombra_9am`, `sombra_mediodia`, `sombra_3pm` | Índice de luz en el solsticio de verano (0 a 255) |
| `dist_horiz_fuego` | Distancia horizontal al punto de ignición de un incendio forestal más cercano, en metros |
| `area_silvestre` | Una de cuatro áreas protegidas (texto) |
| `tipo_suelo` | Uno de 40 tipos de suelo (texto) |

In [ ]:
entrenamiento.head()

In [ ]:
entrenamiento.describe().round(1)

---
## 2. Tu modelo

Tres piezas que puedes cambiar:

- `preparar(df, ref)`: convierte una tabla de parcelas en la matriz de números que recibe la red. `ref`
  es la tabla de la que salen las estadísticas (medias, desviaciones, columnas): siempre la de
  entrenamiento.
- `crear_modelo(n_entradas)`: la arquitectura de la red.
- `EPOCAS`: cuántas épocas entrenar.

Así como vienen funcionan, pero el modelo es flojo: usa solo las columnas numéricas, sin escalar, y
una red diminuta. Ejecútalas tal cual para tu primer envío y después mejóralas.

In [ ]:
NUMERICAS = ["elevacion", "orientacion", "pendiente", "dist_horiz_agua", "dist_vert_agua", "dist_horiz_via",
             "sombra_9am", "sombra_mediodia", "sombra_3pm", "dist_horiz_fuego"]

def preparar(df, ref):
    X = df[NUMERICAS].astype("float32")
    # Ideas: escala las columnas numéricas con la media y la desviación estándar de `ref`.
    #        Convierte area_silvestre y tipo_suelo en columnas de ceros y unos (pd.get_dummies).
    #        Cuidado: la tabla de prueba debe terminar con las mismas columnas, y en el mismo orden,
    #        que la de entrenamiento (mira DataFrame.reindex).
    return X

def crear_modelo(n_entradas):
    modelo = keras.Sequential([
        keras.Input(shape=(n_entradas,)),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(len(CLASES), activation="softmax"),
    ])
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

EPOCAS = 5

### 2.1 Entrenar y validar

Separamos el 20 % del entrenamiento como validación. Las estadísticas de `preparar` salen solo de la
parte de entrenamiento.

In [ ]:
ent, val = train_test_split(entrenamiento, test_size=0.2, stratify=entrenamiento["cobertura"], random_state=42)
a_numero = {c: i for i, c in enumerate(CLASES)}
X_ent, X_val = preparar(ent, ent), preparar(val, ent)
y_ent, y_val = ent["cobertura"].map(a_numero).to_numpy(), val["cobertura"].map(a_numero).to_numpy()

keras.utils.set_random_seed(7)
modelo = crear_modelo(X_ent.shape[1])
historia = modelo.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=EPOCAS, batch_size=64, verbose=0)
curvas(historia, f"{X_ent.shape[1]} columnas · {modelo.count_params():,} parámetros")
print(f"Exactitud de validación en la última época: {historia.history['val_accuracy'][-1]:.1%}")

### 2.2 Predecir la prueba y enviar

Cambia la nota para recordar qué probaste en cada envío. La primera vez te pide la clave.

In [ ]:
X_prueba = preparar(prueba, ent)
predicciones = [CLASES[i] for i in modelo.predict(X_prueba, verbose=0).argmax(1)]
enviar(prueba["id"], predicciones, nota="primer envío")

---
## 3. Pistas para subir en la tabla

Prueba una cosa a la vez, mira la exactitud de **validación** y envía solo cuando mejore.

<details><summary>Pista 1</summary>

Mira `entrenamiento.describe()`: la elevación anda por los miles y las sombras van de 0 a 255. ¿Qué
hicimos con eso en la demostración?
</details>

<details><summary>Pista 2</summary>

`area_silvestre` y `tipo_suelo` son texto y `preparar` las ignora. Un tipo de suelo puede decir mucho
del árbol que crece encima.
</details>

<details><summary>Pista 3</summary>

Con los datos bien preparados, una red más grande, más épocas, `Dropout` y `EarlyStopping` (como en la
demostración) suben varios puntos. Mira las curvas antes de agrandarla más.
</details>

<details><summary>Pista 4</summary>

Cuando ya elegiste tu arquitectura y el número de épocas, puedes entrenar el modelo final con las
14.000 parcelas (entrenamiento más validación) antes de predecir la prueba.
</details>

**Una advertencia.** Si eliges tus cambios mirando solo la tabla en vivo, terminas ajustando el modelo
a esa mitad de la prueba. El resultado final se calcula con la otra mitad.

**Fuente de los datos:** Blackard, J. (1998). *Covertype* [conjunto de datos]. UCI Machine Learning
Repository. https://doi.org/10.24432/C50K5N. Licencia CC BY 4.0. Tomamos una muestra con el mismo
número de parcelas por clase, pasamos las columnas de área y suelo a texto y renombramos todo al español.